# HSE HPC: Standalone Schweinhart Synthetic Validation

This notebook imports the uploaded `schweinhart_controls.py`, which is the exact source file from this repository. It avoids Git and pip bootstrap on the short HSE test allocation.

In [ ]:
from pathlib import Path
import os
import sys

HOME = Path.cwd()
assert (HOME / 'schweinhart_controls.py').exists(), 'Upload schweinhart_controls.py beside this notebook.'
sys.path.insert(0, str(HOME))

import numpy as np
import scipy
from schweinhart_controls import KNOWN_DIMENSIONS, run_schweinhart_controls

print('Working directory:', HOME)
print('CPU count:', os.cpu_count())
print('NumPy:', np.__version__, '| SciPy:', scipy.__version__)

In [ ]:
POINTS = 100_000
SAMPLE_SIZES = [10_000, 20_000, 40_000, 60_000, 80_000, 100_000]
ALPHAS = list(range(1, 11))
SEED = 20260926

controls = run_schweinhart_controls(POINTS, SAMPLE_SIZES, ALPHAS, SEED)
print('Completed deterministic 100,000-point synthetic controls.')

In [ ]:
import pandas as pd

rows = []
for shape, result in controls.items():
    for estimate in result['alpha_estimates']:
        value = estimate['dimension']
        rows.append({
            'shape': shape,
            'known_dimension': result['known_dimension'],
            'alpha': estimate['alpha'],
            'estimated_dimension': value,
            'ape_percent': abs(value - result['known_dimension']) / result['known_dimension'] * 100,
            'r_squared': estimate['r_squared'],
            'ci_95': estimate['dimension_ci_95'],
        })

table = pd.DataFrame(rows)
summary = table.groupby('shape', sort=False).agg(
    known_dimension=('known_dimension', 'first'),
    estimate_min=('estimated_dimension', 'min'),
    estimate_max=('estimated_dimension', 'max'),
    best_ape_percent=('ape_percent', 'min'),
    mean_ape_percent=('ape_percent', 'mean'),
    min_r_squared=('r_squared', 'min'),
)
summary

In [ ]:
assert set(controls) == set(KNOWN_DIMENSIONS)
assert table['estimated_dimension'].notna().all()
assert table['ci_95'].notna().all()
assert (summary['best_ape_percent'] < 5).all()
print('PASS: all four known shapes recover a dimension within 5% at at least one alpha.')